# Extraction latente

> Extraction du latent depuis un `Learner` fastai ou un dataloader PyTorch.


In [ ]:
#| default_exp latent


In [ ]:
#| export
"""Latent-space extraction for fastai learners and generic PyTorch dataloaders."""


from collections.abc import Callable, Iterable
from typing import Any

import torch
from torch import nn

from tell_me_why.callbacks import CollectLatentSpaceCallback, as_fastai_callback
from tell_me_why.config import FastaiLatentResult, LatentBatch
from tell_me_why.tensors import detach_cpu, move_to_device

BatchGetter = Callable[[Any], Any]


def default_input_getter(batch: Any) -> Any:
    """Extract model inputs from common dataloader batch formats."""

    if isinstance(batch, dict):
        for key in ("x", "input", "inputs", "image", "images", "features"):
            if key in batch:
                return batch[key]
        raise KeyError(
            "Could not infer inputs from batch dict. Pass a custom `input_getter`."
        )
    if isinstance(batch, (tuple, list)):
        return batch[0]
    return batch


def default_target_getter(batch: Any) -> Any | None:
    """Extract labels from common dataloader batch formats when available."""

    if isinstance(batch, dict):
        for key in ("y", "label", "labels", "target", "targets"):
            if key in batch:
                return batch[key]
        return None
    if isinstance(batch, (tuple, list)) and len(batch) > 1:
        return batch[1]
    return None


def collect_latents(
    model: nn.Module,
    dataloader: Iterable[Any],
    *,
    device: torch.device | str | None = None,
    input_getter: BatchGetter = default_input_getter,
    target_getter: BatchGetter | None = default_target_getter,
    max_batches: int | None = None,
) -> LatentBatch:
    """Encode batches and concatenate their latent vectors for visualization."""

    was_training = model.training
    model.eval()
    if device is not None:
        model.to(device)

    latents: list[torch.Tensor] = []
    labels: list[torch.Tensor] = []

    try:
        with torch.no_grad():
            for batch_index, batch in enumerate(dataloader):
                if max_batches is not None and batch_index >= max_batches:
                    break

                inputs = input_getter(batch)
                if device is not None:
                    inputs = move_to_device(inputs, device)

                if hasattr(model, "encode"):
                    encoded = model.encode(inputs)
                else:
                    output = model(inputs)
                    encoded = getattr(model, "zi", output)
                latents.append(detach_cpu(encoded))

                if target_getter is not None:
                    target = target_getter(batch)
                    if target is not None:
                        labels.append(detach_cpu(target))
    finally:
        model.train(was_training)

    if not latents:
        raise ValueError("No latent vectors were collected from the dataloader.")

    labels_tensor = torch.cat(labels) if labels else None
    return LatentBatch(latents=torch.cat(latents), labels=labels_tensor)


def extract_latents_from_learner(
    learn: Any,
    *,
    dl: Any | None = None,
    ds_idx: int = 1,
    with_decoded: bool = False,
) -> FastaiLatentResult:
    """Run `learn.get_preds` and return the latent `zi` collected from the AAE."""

    callback = as_fastai_callback(CollectLatentSpaceCallback())
    outputs = learn.get_preds(dl=dl, ds_idx=ds_idx, with_decoded=with_decoded, cbs=[callback])
    preds = outputs[0] if len(outputs) > 0 else None
    targets = outputs[1] if len(outputs) > 1 else None
    latents = getattr(learn, "zi_valid", torch.empty(0))
    vocab = getattr(getattr(learn, "dls", None), "vocab", None)
    return FastaiLatentResult(latents=latents, preds=preds, targets=targets, vocab=vocab)
